INSTALL LIBRARY


In [1]:
!pip install google-play-scraper pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00


IMPORT LIBRARY

In [2]:
import pandas as pd
import re
from wordcloud import WordCloud
import matplotlib.pyplot as plt

from google_play_scraper import reviews, Sort
from urllib.parse import urlparse, parse_qs

In [3]:
import time

WEB SCRAPING

In [4]:
# Link Google Play Store aplikasi wondr by BNI
playstore_url = "https://play.google.com/store/apps/details?id=id.bni.wondr&hl=id"

# Mengambil package name dari link Google Play Store
parsed_url = urlparse(playstore_url)
query_params = parse_qs(parsed_url.query)
package_name = query_params["id"][0]

# Scraping review dari Google Play Store
result, continuation_token = reviews(
    package_name,
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=50000
)

# Mengubah hasil scraping menjadi DataFrame
df = pd.DataFrame(result)

# Mengambil kolom yang dibutuhkan
df = df[['content', 'score', 'at']]

# Mengubah nama kolom
df.columns = ['review', 'rating', 'date']

# Menghapus data kosong pada kolom review
df = df.dropna(subset=['review'])

# Menghapus review yang duplikat
df = df.drop_duplicates(subset=['review'])

# Reset index
df = df.reset_index(drop=True)

HASIL WEB SCRAPING


In [5]:
# Menampilkan informasi dataset
print("Sumber data:", playstore_url)
print("Package name:", package_name)
print("Ukuran dataset:", df.shape)

# Menampilkan 5 data teratas
df.head()

Sumber data: https://play.google.com/store/apps/details?id=id.bni.wondr&hl=id
Package name: id.bni.wondr
Ukuran dataset: (28560, 3)


,review,rating,date
0,woii wondr by BNi kenapa ga bisa transfer si i...,1,2026-07-05 09:56:58
1,ok,5,2026-07-05 09:47:20
2,sat set,5,2026-07-05 09:46:23
3,cepat,5,2026-07-05 09:38:49
4,The Best,5,2026-07-05 09:29:07


SIMPAN KE CSV


In [6]:
df.to_csv("wondr_bni_reviews.csv", index=False, encoding="utf-8-sig")

print("Data hasil scraping berhasil disimpan ke file wondr_bni_reviews.csv")

Data hasil scraping berhasil disimpan ke file wondr_bni_reviews.csv


In [7]:
from google.colab import files
uploaded = files.upload()

Saving wondr_bni_reviews.csv to wondr_bni_reviews (1).csv


PREPROCESSING DATA

In [ ]:

# Install library
!pip install Sastrawi -q

# Import library
import pandas as pd
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Download resource NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# CEK DATAFRAME
try:
    df
except NameError:
    print("ERROR: Dataframe 'df' belum ada.")
    print("Jalankan dulu cell load dataset atau hasil scraping.")
    raise

# CEK KOLOM REVIEW
print("Kolom yang tersedia:")
print(df.columns)

if 'review' not in df.columns:
    raise ValueError(
        "Kolom 'review' tidak ditemukan. "
        "Periksa nama kolom pada dataset."
    )

# STOPWORD DAN STEMMER
stop_words = set(stopwords.words('indonesian'))

factory = StemmerFactory()
stemmer = factory.create_stemmer()

# CLEANING TEXT
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# PREPROCESSING
def preprocess_text(text):
    tokens = word_tokenize(text)

    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    tokens = [
        stemmer.stem(word)
        for word in tokens
    ]

    return " ".join(tokens)

# MENANGANI DATA KOSONG
df['review'] = df['review'].fillna('').astype(str)

# MENJALANKAN PREPROCESSING
print("Memulai preprocessing...")

df['clean_review'] = df['review'].apply(clean_text)
df['clean_review'] = df['clean_review'].apply(preprocess_text)

print("Preprocessing selesai.")

# HASIL PREPROCESSING
df_preprocessing = df[['review', 'clean_review']]

display(df_preprocessing.head(10))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 8.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Kolom yang tersedia:
Index(['review', 'rating', 'date'], dtype='object')
Memulai preprocessing...


LABEL SENTIMEN


In [ ]:
def label_sentiment(score):
    if score >= 4:
        return 'positive'
    elif score == 3:
        return 'neutral'
    else:
        return 'negative'

df['polarity'] = df['rating'].apply(label_sentiment)

print(df['polarity'].value_counts())
df.head()

TF-IDF


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['clean_review'])

y = df['polarity']

SPLIT DATA


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

NAIVE BAYES


In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train, y_train)

PREDIKSI


In [ ]:
y_pred = model.predict(X_test)

AKURASI


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

CLASSIFICATION REPORT

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

DISTRIBUSI SENTIMEN

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

sizes = [count for count in df['polarity'].value_counts()]
labels = list(df['polarity'].value_counts().index)

if len(labels) == 3:
    explode = (0.1, 0, 0)
elif len(labels) == 2:
    explode = (0.1, 0)
else:
    explode = None

plt.pie(
    x=sizes,
    labels=labels,
    autopct='%1.1f%%',
    explode=explode,
    textprops={'fontsize': 14}
)

plt.title('Sentiment Distribution of wondr by BNI App Reviews', fontsize=16, pad=20)
plt.show()

DISTRIBUSI RATING

In [ ]:
import matplotlib.pyplot as plt

# Menghitung jumlah masing-masing rating
rating_count = df['rating'].value_counts().sort_index()

# Membuat Bar Chart
plt.figure(figsize=(8,5))
plt.bar(rating_count.index.astype(str), rating_count.values)

plt.title('Distribusi Rating Pengguna Wondr by BNI')
plt.xlabel('Rating')
plt.ylabel('Jumlah Ulasan')

plt.show()

WORD CLOUD SEMUA REVIEW


In [ ]:
# Semua review menjadi satu teks
all_reviews_text = ' '.join(df['review'].astype(str))

# Bersihkan teks sederhana
all_reviews_text = re.sub(r'http\S+|www\S+', '', all_reviews_text)
all_reviews_text = re.sub(r'[^A-Za-zÀ-ÿ\s]', ' ', all_reviews_text)
all_reviews_text = all_reviews_text.lower()

wordcloud = WordCloud(
    width=900,
    height=500,
    background_color='white',
    min_font_size=10
).generate(all_reviews_text)

plt.figure(figsize=(12, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of wondr by BNI App Reviews', fontsize=18)
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORLD CLOUD POSITIF


In [ ]:
positive_reviews = df[df['polarity'] == 'positive']

# review positif menjadi satu teks
positive_text = ' '.join(positive_reviews['review'].astype(str))

# Bersihkan teks sederhana
positive_text = re.sub(r'http\S+|www\S+', '', positive_text)
positive_text = re.sub(r'[^A-Za-zÀ-ÿ\s]', ' ', positive_text)
positive_text = positive_text.lower()

# Generate WordCloud untuk review positif
wordcloud_positive = WordCloud(
    width=900,
    height=500,
    background_color='white',
    min_font_size=10
).generate(positive_text)

# Tampilkan WordCloud
plt.figure(figsize=(12, 7))
plt.imshow(wordcloud_positive, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Positive wondr by BNI App Reviews', fontsize=18)
plt.show()

WORLD CLOUD NEGATIF


In [ ]:
negative_reviews = df[df['polarity'] == 'negative']

negative_text = ' '.join(negative_reviews['review'].astype(str))

negative_text = re.sub(r'http\S+|www\S+', '', negative_text)
negative_text = re.sub(r'[^A-Za-zÀ-ÿ\s]', ' ', negative_text)
negative_text = negative_text.lower()

wordcloud_negative = WordCloud(
    width=900,
    height=500,
    background_color='white',
    min_font_size=10
).generate(negative_text)

plt.figure(figsize=(12, 7))
plt.imshow(wordcloud_negative, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Negative wondr by BNI App Reviews', fontsize=18)
plt.show()

STREAMLITT


In [ ]:
#DATASET YG AKU PUNYA
print(df.columns.tolist())

1 — HEADER, DATASET, PREPROCESSING, MO

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import re

from collections import Counter
from wordcloud import WordCloud

import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# =====================================
# CONFIG
# =====================================

st.set_page_config(
    page_title="Wondr by BNI Analytics",
    page_icon="📊",
    layout="wide"
)

# =====================================
# CSS
# =====================================

st.markdown("""
<style>

.stApp{
background:linear-gradient(
135deg,
#fff7ed 0%,
#f8fafc 50%,
#ecfeff 100%);
}

.main-title{
font-size:38px;
font-weight:800;
color:#0f172a;
}

.sub-title{
font-size:16px;
color:#475569;
}

.metric-card{
background:white;
padding:20px;
border-radius:18px;
box-shadow:0px 4px 20px rgba(0,0,0,0.08);
text-align:center;
}

</style>
""", unsafe_allow_html=True)

# =====================================
# LOAD DATA
# =====================================

df = pd.read_csv("wondr_bni_reviews.csv")

df.columns = df.columns.str.strip()

df["review"] = df["review"].fillna("").astype(str)

# =====================================
# LABEL SENTIMEN
# =====================================

def sentiment_label(rating):

    if rating >= 4:
        return "positive"

    elif rating == 3:
        return "neutral"

    else:
        return "negative"

if "polarity" not in df.columns:
    df["polarity"] = df["rating"].apply(sentiment_label)

# =====================================
# TF-IDF
# =====================================

tfidf = TfidfVectorizer(max_features=3000)

df["clean_review"] = df["clean_review"].fillna("").astype(str)

X = tfidf.fit_transform(df["clean_review"])
X = tfidf.fit_transform(df["clean_review"])

y = df["polarity"]

# =====================================
# SPLIT
# =====================================

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================
# NAIVE BAYES
# =====================================

model = MultinomialNB()

model.fit(X_train,y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)

df["predicted_sentiment"] = model.predict(X)

df["review_length"] = df["review"].apply(len)

with st.sidebar:

    st.title("📊 Wondr Analytics")

    menu = st.radio(
        "Menu",
        [
            "Dashboard",
            "Dataset",
            "Analisis Sentimen",
            "WordCloud",
            "Prediksi Sentimen"
        ]
    )

# =====================================
# DASHBOARD
# =====================================

if menu == "Dashboard":

    st.markdown(
        "<div class='main-title'>📊 Wondr by BNI Analytics</div>",
        unsafe_allow_html=True
    )

    st.markdown(
        "<div class='sub-title'>Analisis Sentimen Ulasan Pengguna Menggunakan Naive Bayes</div>",
        unsafe_allow_html=True
    )

    st.write("")

    total_review = len(df)

    avg_rating = round(df["rating"].mean(),2)

    dominant_sentiment = (
        df["polarity"]
        .value_counts()
        .idxmax()
    )

    c1,c2,c3,c4 = st.columns(4)

    c1.metric("Total Review",f"{total_review:,}")

    c2.metric("Rata-rata Rating",avg_rating)

    c3.metric(
        "Akurasi Model",
        f"{accuracy*100:.2f}%"
    )

    c4.metric(
        "Sentimen Dominan",
        dominant_sentiment
    )

    st.divider()

    col1,col2 = st.columns(2)

    with col1:

        sentiment_count = (
            df["polarity"]
            .value_counts()
            .reset_index()
        )

        sentiment_count.columns = [
            "Sentiment",
            "Jumlah"
        ]

        fig = px.pie(
            sentiment_count,
            names="Sentiment",
            values="Jumlah",
            hole=0.4,
            title="Distribusi Sentimen"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    with col2:

        rating_count = (
            df["rating"]
            .value_counts()
            .sort_index()
            .reset_index()
        )

        rating_count.columns = [
            "Rating",
            "Jumlah"
        ]

        fig2 = px.bar(
            rating_count,
            x="Rating",
            y="Jumlah",
            title="Distribusi Rating"
        )

        st.plotly_chart(
            fig2,
            use_container_width=True
        )

# =====================================
# DATASET
# =====================================

elif menu == "Dataset":

    st.subheader("📄 Dataset Ulasan")

    st.dataframe(
        df[
            [
                "review",
                "rating",
                "polarity"
            ]
        ],
        use_container_width=True
    )

# =====================================
# ANALISIS SENTIMEN
# =====================================

elif menu == "Analisis Sentimen":

    st.subheader("📊 Analisis Sentimen")

    sentiment_count = (
        df["polarity"]
        .value_counts()
        .reset_index()
    )

    sentiment_count.columns = [
        "Sentiment",
        "Jumlah"
    ]

    fig = px.bar(
        sentiment_count,
        x="Sentiment",
        y="Jumlah",
        color="Sentiment"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

# =====================================
# WORDCLOUD
# =====================================

elif menu == "WordCloud":

    st.subheader("☁️ WordCloud")

    text = " ".join(
        df["clean_review"]
        .astype(str)
        .tolist()
    )

    wordcloud = WordCloud(
        width=1200,
        height=500,
        background_color="white"
    ).generate(text)

    fig,ax = plt.subplots(
        figsize=(12,5)
    )

    ax.imshow(wordcloud)

    ax.axis("off")

    st.pyplot(fig)

    words = re.findall(
        r"\b[a-zA-Z]+\b",
        text
    )

    freq = Counter(words)

    top_words = pd.DataFrame(
        freq.most_common(20),
        columns=[
            "Kata",
            "Frekuensi"
        ]
    )

    st.dataframe(
        top_words,
        use_container_width=True
    )

# =====================================
# PREDIKSI
# =====================================

elif menu == "Prediksi Sentimen":

    st.subheader(
        "🤖 Prediksi Sentimen"
    )

    review = st.text_area(
        "Masukkan Ulasan"
    )

    if st.button("Prediksi"):

        review = review.lower()

        vector = tfidf.transform(
            [review]
        )

        prediksi = model.predict(
            vector
        )[0]

        if prediksi == "positive":

            st.success(
                "😊 Sentimen Positif"
            )

        elif prediksi == "neutral":

            st.warning(
                "😐 Sentimen Netral"
            )

        else:

            st.error(
                "😞 Sentimen Negatif"
            )

STREAMLIT PAKE NGROK


In [ ]:
!pip install streamlit pyngrok -q
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!cat /content/logs.txt

In [ ]:
!ngrok config add-authtoken 37pYnULttTF31X4VWMcO71OjMkr_4sc8unc92TA3Qy2MmYVqa

In [ ]:
print(df.columns.tolist())

In [ ]:
df.to_csv("wondr_bni_reviews.csv", index=False)

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok
print(ngrok.connect(8501))